In [1]:
!pip install -q timm fvcore

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import timm
import pandas as pd
from fvcore.nn import FlopCountAnalysis

# 1. Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

assert torch.cuda.is_available(), "Switch your Colab runtime to T4 GPU first!"
device_cpu = torch.device("cpu")
device_gpu = torch.device("cuda")

# 2. Load Held-Out Dataset (Imagenette Validation Set)
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = torchvision.datasets.Imagenette(
    root="./data", split="val", size="320px", download=True, transform=transform
)
sample_img, sample_label = dataset[0]
img_input = sample_img.unsqueeze(0)  # Shape: (1, 3, 256, 256)
print(f"Loaded Imagenette val set: {len(dataset)} images | Sample shape: {tuple(img_input.shape)}\n")

# 3. Define Models
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=128, hidden_size=256, num_layers=2, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

seq_input = torch.randn(1, 128, 128)  # (Batch=1, Seq_Len=128, Features=128)

models_dict = {
    "MobileViT-S": (timm.create_model("mobilevit_s", pretrained=True).eval(), img_input),
    "MobileNetV2": (timm.create_model("mobilenetv2_100", pretrained=True).eval(), img_input),
    "LSTM (2-Layer)": (LSTMClassifier().eval(), seq_input),
}

# 4. Profile Params, FLOPs, and Validate CPU vs. GPU Outputs
results = []

for name, (model, sample_in) in models_dict.items():
    # Parameter count
    params = sum(p.numel() for p in model.parameters())

    # FLOPs count (fvcore counts MACs; 1 MAC = 2 FLOPs)
    model_cpu = model.to(device_cpu)
    flops_analyzer = FlopCountAnalysis(model_cpu, sample_in)
    flops_analyzer.unsupported_ops_warnings(False)
    macs = flops_analyzer.total()
    flops = macs * 2

    # CPU vs. GPU Output Validation (atol=1e-3)
    with torch.no_grad():
        out_cpu = model_cpu(sample_in)
        model_gpu = model.to(device_gpu)
        out_gpu = model_gpu(sample_in.to(device_gpu)).cpu()

    max_diff = (out_cpu - out_gpu).abs().max().item()
    passed = torch.allclose(out_cpu, out_gpu, atol=1e-3)

    results.append({
        "Model": name,
        "Input_Shape": str(tuple(sample_in.shape)),
        "Params_M": round(params / 1e6, 3),
        "MACs_G": round(macs / 1e9, 4),
        "FLOPs_G": round(flops / 1e9, 4),
        "Max_Abs_Diff": f"{max_diff:.2e}",
        "Match_atol_1e-3": passed,
    })

    # Move back to CPU to keep GPU memory clean
    model.to(device_cpu)
    torch.cuda.empty_cache()

specs_df = pd.DataFrame(results)
specs_df.to_csv("model_specs.csv", index=False)
print(specs_df.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
100%|██████████| 342M/342M [00:08<00:00, 42.5MB/s]


Loaded Imagenette val set: 3925 images | Sample shape: (1, 3, 256, 256)



model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

stages.2.1.transformer.0.attn.attn_drop, stages.2.1.transformer.1.attn.attn_drop, stages.3.1.transformer.0.attn.attn_drop, stages.3.1.transformer.1.attn.attn_drop, stages.3.1.transformer.2.attn.attn_drop, stages.3.1.transformer.3.attn.attn_drop, stages.4.1.transformer.0.attn.attn_drop, stages.4.1.transformer.1.attn.attn_drop, stages.4.1.transformer.2.attn.attn_drop


         Model      Input_Shape  Params_M  MACs_G  FLOPs_G Max_Abs_Diff  Match_atol_1e-3
   MobileViT-S (1, 3, 256, 256)     5.579  1.8558   3.7116     1.22e-03             True
   MobileNetV2 (1, 3, 256, 256)     3.505  0.4100   0.8200     4.98e-06             True
LSTM (2-Layer)    (1, 128, 128)     0.924  0.0000   0.0000     7.45e-09             True


In [3]:
# Analytical MACs/FLOPs for 2-layer LSTM (T=128, I=128, H=256, C=10)
T, I, H, C = 128, 128, 256, 10
lstm_macs = T * 4 * (I + H) * H + T * 4 * (H + H) * H + H * C
lstm_flops = lstm_macs * 2

specs_df.loc[specs_df["Model"] == "LSTM (2-Layer)", "MACs_G"] = round(lstm_macs / 1e9, 4)
specs_df.loc[specs_df["Model"] == "LSTM (2-Layer)", "FLOPs_G"] = round(lstm_flops / 1e9, 4)

specs_df.to_csv("model_specs.csv", index=False)
print(specs_df.to_string(index=False))

         Model      Input_Shape  Params_M  MACs_G  FLOPs_G Max_Abs_Diff  Match_atol_1e-3
   MobileViT-S (1, 3, 256, 256)     5.579  1.8558   3.7116     1.22e-03             True
   MobileNetV2 (1, 3, 256, 256)     3.505  0.4100   0.8200     4.98e-06             True
LSTM (2-Layer)    (1, 128, 128)     0.924  0.1174   0.2349     7.45e-09             True
